## Trajectory Candidate Study: planner vs human prototypes vs external clusters

This notebook helps you study **"human actions"** in a processed dataset (executed motion from `odom.jsonl`) and compare candidate-trajectory sets that a VLM could select from in **Task 2**.

We compare:
- **Planner candidates** (current baseline): `episodes/*/planner_candidates.jsonl` (robot-frame trajectories)
- **Human-derived prototypes**: cluster executed future trajectories from `odom.jsonl` (robot-frame) via simple KMeans
- **Grid prototypes**: simple hand-designed endpoint grid (optional)
- **External prototypes**: `data/external/extended_web_scale_cluster_center.npy` (optional)

Key idea: turn everything into the same representation: a trajectory = `T x 2` array of `(x, y)` in **robot frame**.

---

### What you should edit first
- `DATASET_ROOT_BASE` (your processed dataset with prelogged candidates)
- `DATASET_ROOT_GOALLESS` (optional: goal-less ONNX candidate dataset)
- `USE_TAKEOVER_CLIPS` / `TAKEOVER_CLIPS_PATH` (recommended to match Task 2 evaluation times)
- `EXTERNAL_NPY_PATH` (points to the external cluster file, if accessible)


In [ ]:
import json
import math
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

# --- User config ---

# Base dataset (goal-conditioned / prelogged candidates)
DATASET_ROOT_BASE = Path("data/processed")

# Goal-less ONNX candidates generated at clip times by the planner module.
DATASET_ROOT_GOALLESS = Path("data/processed_goal_less_onnx")

# Evaluate on the same snapshot protocol as Task 2 (recommended)
USE_TAKEOVER_CLIPS = True
TAKEOVER_CLIPS_PATH = DATASET_ROOT_BASE / "takeover_clips" / "takeover_clips.jsonl"
TAKEOVER_CLIPS_TIME_KEY = "t0"  # matches takeover_clips.jsonl schema
TIME_TOL_S = 0.21  # time matching tolerance when mapping clip t0 -> odom/candidate timestamps

# Optional external prototype file
EXTERNAL_NPY_PATH = Path("data/external/extended_web_scale_cluster_center.npy")

# Trajectory representation: use the same length as planner candidates (default: 20 points)
N_POINTS = 20
DT_S = 0.2  # from process_summary.json; odom timestamps are sample_index * dt

# Sampling fallback (only used when USE_TAKEOVER_CLIPS=False)
MAX_EPISODES = None  # e.g. 5 for quick iteration
SAMPLE_STRIDE = 5  # take every N-th tick within each episode
MAX_SAMPLES_PER_EPISODE = 200

# Prototypes
K_HUMAN_KMEANS = 64
K_GRID = 64

# Candidate sources to compare (A/B)
CANDIDATE_SOURCES = {"prelogged": DATASET_ROOT_BASE}
if DATASET_ROOT_GOALLESS.exists():
    CANDIDATE_SOURCES["goal_less_onnx"] = DATASET_ROOT_GOALLESS
else:
    print(f"Optional goal-less ONNX dataset not found; skipping it: {DATASET_ROOT_GOALLESS}")

np.random.seed(0)
plt.rcParams["figure.figsize"] = (7, 6)
plt.rcParams["axes.grid"] = True

assert DATASET_ROOT_BASE.exists(), f"Dataset root not found: {DATASET_ROOT_BASE}"
if USE_TAKEOVER_CLIPS:
    assert TAKEOVER_CLIPS_PATH.exists(), f"Missing takeover_clips.jsonl: {TAKEOVER_CLIPS_PATH}"

In [ ]:
def read_jsonl(path: Path):
    with path.open("r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)


def load_episode_ids(dataset_root: Path):
    manifest = dataset_root / "dataset_manifest.json"
    if manifest.exists():
        obj = json.loads(manifest.read_text())
        return list(obj["episodes"])

    # fallback: list directory
    eps_dir = dataset_root / "episodes"
    return sorted([p.name for p in eps_dir.iterdir() if p.is_dir()])


def load_odom_arrays(episode_dir: Path):
    """Return t,x,y,yaw arrays from odom.jsonl (map_enu frame)."""
    odom_path = episode_dir / "odom.jsonl"
    t_list, x_list, y_list, yaw_list = [], [], [], []
    for rec in read_jsonl(odom_path):
        t_list.append(float(rec["t"]))
        x_list.append(float(rec["x"]))
        y_list.append(float(rec["y"]))
        yaw_list.append(float(rec["yaw"]))
    return (
        np.asarray(t_list, dtype=np.float32),
        np.asarray(x_list, dtype=np.float32),
        np.asarray(y_list, dtype=np.float32),
        np.asarray(yaw_list, dtype=np.float32),
    )


def world_to_robot(dx: np.ndarray, dy: np.ndarray, yaw0: float):
    """Rotate world-frame displacement into robot frame at yaw0.

    We assume yaw0 is the robot heading in the world/map frame.

    Robot frame convention used here:
    - x: forward
    - y: left
    """
    c = math.cos(-yaw0)
    s = math.sin(-yaw0)
    x_r = c * dx - s * dy
    y_r = s * dx + c * dy
    return x_r, y_r


def executed_future_traj_robot(x, y, yaw, idx0: int, n_points: int):
    """Executed future trajectory in robot frame at idx0.

    Returns shape (n_points, 2) for steps 1..n_points.
    """
    if idx0 + n_points >= len(x):
        return None

    x0, y0, yaw0 = float(x[idx0]), float(y[idx0]), float(yaw[idx0])
    steps = np.arange(1, n_points + 1, dtype=np.int64)

    dx = (x[idx0 + steps] - x0).astype(np.float32)
    dy = (y[idx0 + steps] - y0).astype(np.float32)

    xr, yr = world_to_robot(dx, dy, yaw0)
    return np.stack([xr, yr], axis=1)


def ade(traj_a: np.ndarray, traj_b: np.ndarray):
    """Average Displacement Error between two (T,2) trajectories."""
    d = np.linalg.norm(traj_a - traj_b, axis=1)
    return float(d.mean())


def minade(traj: np.ndarray, candidates: np.ndarray):
    """Minimum ADE between a (T,2) traj and candidate set (K,T,2)."""
    # vectorized: (K,T,2) - (T,2) -> (K,T)
    d = np.linalg.norm(candidates - traj[None, :, :], axis=2)
    return float(d.mean(axis=1).min())


def plot_trajs(ax, trajs, color="C0", alpha=0.2, lw=1.0, label=None):
    first = True
    for tr in trajs:
        ax.plot(tr[:, 0], tr[:, 1], color=color, alpha=alpha, lw=lw, label=label if first else None)
        first = False


def set_equal_aspect(ax):
    ax.set_aspect("equal", adjustable="box")
    ax.axhline(0.0, color="k", lw=0.5)
    ax.axvline(0.0, color="k", lw=0.5)
    ax.set_xlabel("x (forward)")
    ax.set_ylabel("y (left)")

In [ ]:
def load_times_by_episode_from_jsonl(path: Path, *, time_key: str = "t0") -> dict[str, list[float]]:
    out: dict[str, list[float]] = {}
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception:
                continue
            if not isinstance(obj, dict):
                continue
            eid = obj.get("episode_id")
            tt = obj.get(time_key)
            if not isinstance(eid, str) or not isinstance(tt, (int, float)):
                continue
            out.setdefault(str(eid), []).append(float(tt))

    for eid in list(out.keys()):
        out[eid] = sorted(out[eid])
    return out


def _nearest_index(times: np.ndarray, t0: float, *, tol_s: float) -> int | None:
    if times.size == 0:
        return None
    tt = float(t0)
    j = int(np.searchsorted(times, tt, side="left"))
    cand = []
    if 0 <= j < int(times.size):
        cand.append(j)
    if 0 <= j - 1 < int(times.size):
        cand.append(j - 1)
    best = None
    best_dt = None
    for k in cand:
        dt = abs(float(times[k]) - tt)
        if best_dt is None or dt < best_dt:
            best_dt = dt
            best = int(k)
    if best is None:
        return None
    if best_dt is not None and float(best_dt) > float(tol_s):
        return None
    return int(best)


def load_candidates_at_times(
    planner_candidates_path: Path, wanted_times: set[float], *, n_points: int
):
    """Return dict: rounded_t -> {"trajs": (K,T,2), "scores": (K,)}"""
    out: dict[float, dict[str, Any]] = {}
    if not wanted_times:
        return out

    with planner_candidates_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except Exception:
                continue
            if not isinstance(rec, dict):
                continue
            try:
                tt = round(float(rec.get("t", 0.0)), 6)
            except Exception:
                continue
            if tt not in wanted_times:
                continue

            cand = rec.get("candidates") or []
            trajs = []
            scores = []
            for c in cand:
                if not isinstance(c, dict):
                    continue
                pts = np.asarray(c.get("points_xy") or [], dtype=np.float32)
                if pts.ndim != 2 or pts.shape[1] < 2:
                    continue
                trajs.append(pts[:n_points, :2])
                try:
                    scores.append(float(c.get("score", 0.0)))
                except Exception:
                    scores.append(0.0)

            if not trajs:
                continue

            out[tt] = {
                "trajs": np.stack(trajs, axis=0),
                "scores": np.asarray(scores, dtype=np.float32),
            }

    return out


def collect_dataset_samples_multi(
    *,
    dataset_root_odom: Path,
    candidate_sources: dict[str, Path],
    n_points: int,
    max_episodes=None,
    sample_stride: int = 5,
    max_samples_per_episode: int = 200,
    use_takeover_clips: bool = True,
    takeover_clips_path: Path | None = None,
    takeover_clips_time_key: str = "t0",
    time_tol_s: float = 0.21,
):
    """Collect executed trajectories + candidates from multiple sources, aligned by
    (episode_id, t0).

    Returns:
      executed_trajs: (N,T,2)
      candidates: dict[name -> {"trajs": (N,K,T,2), "scores": (N,K)}]
      meta: list[dict]
    """

    dataset_root_odom = Path(dataset_root_odom)
    episode_ids = load_episode_ids(dataset_root_odom)
    if max_episodes is not None:
        episode_ids = episode_ids[: int(max_episodes)]

    clips_by_ep: dict[str, list[float]] = {}
    if use_takeover_clips:
        if takeover_clips_path is None:
            raise ValueError("use_takeover_clips=True but takeover_clips_path is None")
        clips_by_ep = load_times_by_episode_from_jsonl(
            Path(takeover_clips_path), time_key=str(takeover_clips_time_key)
        )

    executed_trajs: list[np.ndarray] = []
    meta: list[dict] = []
    cand_trajs: dict[str, list[np.ndarray]] = {k: [] for k in candidate_sources.keys()}
    cand_scores: dict[str, list[np.ndarray]] = {k: [] for k in candidate_sources.keys()}

    K_ref: int | None = None

    for ep in tqdm(episode_ids, desc="episodes"):
        ep_dir_odom = dataset_root_odom / "episodes" / ep
        t, x, y, yaw = load_odom_arrays(ep_dir_odom)

        # Decide which times to evaluate.
        times: list[float] = []
        if use_takeover_clips:
            times = list(clips_by_ep.get(ep, []))
        else:
            valid_max = len(t) - (n_points + 1)
            if valid_max > 0:
                idxs = list(range(0, valid_max, int(sample_stride)))
                if max_samples_per_episode is not None:
                    idxs = idxs[: int(max_samples_per_episode)]
                times = [float(t[i]) for i in idxs]

        if not times:
            continue

        wanted_times = set(round(float(tt), 6) for tt in times)

        # Load candidate maps for each source (time->candidates). We filter while reading for speed.
        cand_by_src: dict[str, dict[float, dict[str, Any]]] = {}
        for name, root in candidate_sources.items():
            pc = Path(root) / "episodes" / ep / "planner_candidates.jsonl"
            if not pc.exists():
                cand_by_src[name] = {}
                continue
            cand_by_src[name] = load_candidates_at_times(pc, wanted_times, n_points=n_points)

        for tt in times:
            tt6 = round(float(tt), 6)

            # Map time->odom index
            idx0 = _nearest_index(t.astype(np.float64), float(tt), tol_s=float(time_tol_s))
            if idx0 is None:
                continue
            tr = executed_future_traj_robot(x, y, yaw, int(idx0), int(n_points))
            if tr is None:
                continue

            # Require that ALL candidate sources have this time (so arrays align for A/B).
            payloads = {}
            ok = True
            for name in candidate_sources.keys():
                payload = cand_by_src.get(name, {}).get(tt6)
                if payload is None:
                    ok = False
                    break
                payloads[name] = payload
            if not ok:
                continue

            # Validate and append.
            # (We keep only samples where all sources have consistent K.)
            k_this: int | None = None
            for payload in payloads.values():
                trajs = payload.get("trajs")
                scores = payload.get("scores")
                if trajs is None or scores is None:
                    ok = False
                    break
                if trajs.ndim != 3 or trajs.shape[1] != int(n_points):
                    ok = False
                    break
                if scores.ndim != 1 or scores.shape[0] != trajs.shape[0]:
                    ok = False
                    break
                k_this = int(trajs.shape[0]) if k_this is None else k_this
                if int(trajs.shape[0]) != int(k_this):
                    ok = False
                    break
            if not ok:
                continue

            if K_ref is None:
                K_ref = int(k_this or 0)
            if int(k_this or 0) != int(K_ref):
                continue

            executed_trajs.append(tr)
            meta.append({"episode_id": ep, "idx0": int(idx0), "t0": float(tt)})
            for name, payload in payloads.items():
                cand_trajs[name].append(payload["trajs"])
                cand_scores[name].append(payload["scores"])

    executed_arr = np.stack(executed_trajs, axis=0)
    out: dict[str, dict[str, Any]] = {}
    for name in candidate_sources.keys():
        out[name] = {
            "trajs": np.stack(cand_trajs[name], axis=0),
            "scores": np.stack(cand_scores[name], axis=0),
        }
    return executed_arr, out, meta


# Collect aligned samples (recommended: takeover-clip times, matching Task 2).
executed_trajs, cand_by_src, meta = collect_dataset_samples_multi(
    dataset_root_odom=DATASET_ROOT_BASE,
    candidate_sources=CANDIDATE_SOURCES,
    n_points=N_POINTS,
    max_episodes=MAX_EPISODES,
    sample_stride=SAMPLE_STRIDE,
    max_samples_per_episode=MAX_SAMPLES_PER_EPISODE,
    use_takeover_clips=bool(USE_TAKEOVER_CLIPS),
    takeover_clips_path=TAKEOVER_CLIPS_PATH,
    takeover_clips_time_key=TAKEOVER_CLIPS_TIME_KEY,
    time_tol_s=TIME_TOL_S,
)

planner_sets = cand_by_src["prelogged"]["trajs"]
planner_scores = cand_by_src["prelogged"]["scores"]

goal_less_payload = cand_by_src.get("goal_less_onnx")
planner_sets_goal_less = goal_less_payload["trajs"] if goal_less_payload else None
planner_scores_goal_less = goal_less_payload["scores"] if goal_less_payload else None

print("executed_trajs:", executed_trajs.shape)
print("prelogged planner_sets:", planner_sets.shape, planner_scores.shape)
if planner_sets_goal_less is not None and planner_scores_goal_less is not None:
    print("goal_less planner_sets:", planner_sets_goal_less.shape, planner_scores_goal_less.shape)
print("N episodes in meta:", len(set(m["episode_id"] for m in meta)))
print("N samples (kept):", len(meta))

if USE_TAKEOVER_CLIPS:
    clips_by_ep = load_times_by_episode_from_jsonl(
        TAKEOVER_CLIPS_PATH, time_key=TAKEOVER_CLIPS_TIME_KEY
    )
    total_clips = sum(len(v) for v in clips_by_ep.values())
    print("N takeover clips (total):", total_clips)
    print(
        "Note: kept samples can be < total clips if near episode end "
        "(insufficient future horizon) or time-match fails."
    )

In [ ]:
# --- Quick sanity plots: executed trajectory distribution ---

N = executed_trajs.shape[0]

# Random overlays
n_show = min(200, N)
idxs = np.random.choice(N, size=n_show, replace=False)

fig, ax = plt.subplots()
plot_trajs(ax, executed_trajs[idxs], color="C0", alpha=0.15, lw=1.0, label="executed")
ax.set_title(f"Executed future trajectories (random {n_show}/{N})")
set_equal_aspect(ax)
ax.legend()
plt.show()

# Endpoint scatter
endpoints = executed_trajs[:, -1, :]
fig, ax = plt.subplots()
ax.scatter(endpoints[:, 0], endpoints[:, 1], s=5, alpha=0.3)
ax.set_title("Executed endpoints (last point)")
set_equal_aspect(ax)
plt.show()

# Endpoint density heatmap (2D hist)
fig, ax = plt.subplots()
H, xedges, yedges = np.histogram2d(endpoints[:, 0], endpoints[:, 1], bins=60)
ax.imshow(
    H.T,
    origin="lower",
    extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
    aspect="equal",
)
ax.set_title("Executed endpoint density (2D hist)")
set_equal_aspect(ax)
plt.show()

In [ ]:
def kmeans_numpy(X: np.ndarray, k: int, n_iter: int = 50, seed: int = 0):
    """Very small KMeans (L2) implementation to avoid extra deps.

    X: (N,D)
    Returns: centers (k,D), labels (N,)
    """
    rng = np.random.default_rng(seed)
    N, D = X.shape

    # kmeans++ init
    centers = np.empty((k, D), dtype=np.float32)
    i0 = rng.integers(0, N)
    centers[0] = X[i0]

    d2 = np.sum((X - centers[0]) ** 2, axis=1)
    for ci in range(1, k):
        probs = d2 / (d2.sum() + 1e-12)
        idx = rng.choice(N, p=probs)
        centers[ci] = X[idx]
        d2 = np.minimum(d2, np.sum((X - centers[ci]) ** 2, axis=1))

    labels = np.zeros((N,), dtype=np.int64)

    for _ in range(n_iter):
        # assign
        # distances: (N,k)
        dist2 = np.sum((X[:, None, :] - centers[None, :, :]) ** 2, axis=2)
        new_labels = dist2.argmin(axis=1)

        if np.array_equal(new_labels, labels):
            break
        labels = new_labels

        # update
        for ci in range(k):
            mask = labels == ci
            if not np.any(mask):
                # re-seed empty cluster
                centers[ci] = X[rng.integers(0, N)]
            else:
                centers[ci] = X[mask].mean(axis=0)

    return centers.astype(np.float32), labels


# Flatten (T,2) -> (2T)
X = executed_trajs.reshape(executed_trajs.shape[0], -1)
centers_flat, labels = kmeans_numpy(X, k=K_HUMAN_KMEANS, n_iter=60, seed=0)
human_centers = centers_flat.reshape(K_HUMAN_KMEANS, N_POINTS, 2)

# Plot a subset of cluster centers
order = np.bincount(labels, minlength=K_HUMAN_KMEANS).argsort()[::-1]

n_show = min(24, K_HUMAN_KMEANS)
fig, ax = plt.subplots()
plot_trajs(
    ax, human_centers[order[:n_show]], color="C3", alpha=0.8, lw=2.0, label="human kmeans centers"
)
ax.set_title(f"Top {n_show} human KMeans trajectory centers (by frequency)")
set_equal_aspect(ax)
ax.legend()
plt.show()

In [ ]:
def make_endpoint_grid_from_data(endpoints_xy: np.ndarray, k: int):
    """Make a simple grid of endpoints based on data quantiles."""
    side = int(round(math.sqrt(k)))
    side = max(2, side)
    k_eff = side * side

    # Use robust ranges (5%..95%) to avoid extreme outliers
    xs = np.quantile(endpoints_xy[:, 0], np.linspace(0.05, 0.95, side))
    ys = np.quantile(endpoints_xy[:, 1], np.linspace(0.05, 0.95, side))

    grid = np.array([(x, y) for x in xs for y in ys], dtype=np.float32)
    return grid, k_eff


def generate_smooth_traj_to_endpoint(endpoint_xy: np.ndarray, n_points: int):
    """Simple smooth-ish trajectory from (0,0) to (x_end,y_end).

    This is not a dynamics-feasible model; it is just a baseline "grid" generator.
    """
    x_end, y_end = float(endpoint_xy[0]), float(endpoint_xy[1])
    s = np.linspace(0.0, 1.0, n_points + 1, dtype=np.float32)[1:]  # exclude 0

    x = s * x_end
    # Use s^2 so lateral starts gently
    y = (s**2) * y_end
    return np.stack([x, y], axis=1)


def make_grid_prototypes(endpoints_xy: np.ndarray, k: int, n_points: int):
    grid_endpoints, k_eff = make_endpoint_grid_from_data(endpoints_xy, k)
    trajs = np.stack(
        [generate_smooth_traj_to_endpoint(ep, n_points) for ep in grid_endpoints], axis=0
    )
    return trajs, grid_endpoints, k_eff


grid_protos, grid_endpoints, k_eff = make_grid_prototypes(endpoints, K_GRID, N_POINTS)
print("grid_protos:", grid_protos.shape, "(k_eff=", k_eff, ")")

fig, ax = plt.subplots()
plot_trajs(
    ax, grid_protos[: min(64, len(grid_protos))], color="C2", alpha=0.4, lw=1.5, label="grid protos"
)
ax.set_title("Grid prototypes (subset)")
set_equal_aspect(ax)
ax.legend()
plt.show()

In [ ]:
def load_external_npy_prototypes(path: Path, n_points: int | None = None):
    """Load external cluster centers and try to coerce into (K,T,2).

    De-normalization rule for the exported cluster centers:
    - x = x * 0.51
    - y = y * 0.32 - 0.16
    - values were produced in [0,1] space
    """
    arr = np.load(path)

    # Try to interpret shape
    if arr.ndim == 3 and arr.shape[-1] == 2:
        trajs = arr.astype(np.float32)
    elif arr.ndim == 2 and arr.shape[1] % 2 == 0:
        T = arr.shape[1] // 2
        trajs = arr.reshape(arr.shape[0], T, 2).astype(np.float32)
    else:
        raise ValueError(f"Unsupported npy shape {arr.shape}; expected (K,T,2) or (K,2T)")

    if n_points is not None:
        trajs = trajs[:, :n_points, :]

    trajs = trajs.copy()
    trajs[:, :, 0] = trajs[:, :, 0] * 0.51
    trajs[:, :, 1] = trajs[:, :, 1] * 0.32 - 0.16

    trajs[..., 0] = trajs[..., 0].cumsum(axis=1)
    trajs[..., 1] = trajs[..., 1].cumsum(axis=1)

    return trajs


external_protos = None
if EXTERNAL_NPY_PATH.exists():
    external_protos = load_external_npy_prototypes(EXTERNAL_NPY_PATH, n_points=N_POINTS)
    print("external_protos:", external_protos.shape)

    # --- DEBUGGING: print example trajectory to check for weird values ---
    print("external_protos min/max:", external_protos.min(), external_protos.max())
    # print("external_protos[0]:\n", external_protos[0])

    # Check for any NaNs or infs
    if np.isnan(external_protos).any():
        print("Warning: NaNs in external_protos!")
    if np.isinf(external_protos).any():
        print("Warning: infs in external_protos!")

    # Proper plotting (should plot each (T,2) as a line) --
    fig, ax = plt.subplots(figsize=(7, 6))
    for i in range(external_protos.shape[0]):
        xs = external_protos[i, :, 0]
        ys = external_protos[i, :, 1]
        print(
            f"Mode {i}: shape {xs.shape}, X-min {xs.min():.3f}, X-max {xs.max():.3f}, "
            f"Y-min {ys.min():.3f}, Y-max {ys.max():.3f}"
        )
        ax.plot(xs, ys, color="C1", alpha=0.35, lw=1.5)
    ax.set_title(f"External npy prototypes ({external_protos.shape[0]} trajectories)")
    set_equal_aspect(ax)
    plt.show()
else:
    print(f"External npy not found (skipping): {EXTERNAL_NPY_PATH}")

In [ ]:
external_protos.shape

In [ ]:
def minade_fixed_candidates(executed: np.ndarray, candidates: np.ndarray, chunk_size: int = 512):
    """executed: (N,T,2), candidates: (K,T,2) -> (N,) minADE."""
    N = executed.shape[0]
    out = np.empty((N,), dtype=np.float32)
    for i0 in range(0, N, chunk_size):
        i1 = min(N, i0 + chunk_size)
        ex = executed[i0:i1]  # (B,T,2)
        d = np.linalg.norm(candidates[None, :, :, :] - ex[:, None, :, :], axis=3)  # (B,K,T)
        out[i0:i1] = d.mean(axis=2).min(axis=1)
    return out


def minade_per_sample_candidates(
    executed: np.ndarray, candidates_per_sample: np.ndarray, chunk_size: int = 256
):
    """executed: (N,T,2), candidates_per_sample: (N,K,T,2) -> (N,) minADE."""
    N = executed.shape[0]
    out = np.empty((N,), dtype=np.float32)
    for i0 in range(0, N, chunk_size):
        i1 = min(N, i0 + chunk_size)
        ex = executed[i0:i1]  # (B,T,2)
        cand = candidates_per_sample[i0:i1]  # (B,K,T,2)
        d = np.linalg.norm(cand - ex[:, None, :, :], axis=3)  # (B,K,T)
        out[i0:i1] = d.mean(axis=2).min(axis=1)
    return out


def plot_cdf(ax, values: np.ndarray, label: str):
    v = np.sort(values)
    y = np.linspace(0.0, 1.0, len(v), endpoint=True)
    ax.plot(v, y, label=label)


# --- Coverage comparison: how well does each candidate set "cover" executed trajectories? ---

minade_planner = minade_per_sample_candidates(executed_trajs, planner_sets)
minade_human_kmeans = minade_fixed_candidates(executed_trajs, human_centers)
minade_grid = minade_fixed_candidates(executed_trajs, grid_protos)

minade_external = None
if external_protos is not None:
    minade_external = minade_fixed_candidates(executed_trajs, external_protos)

snapshot_mode = "takeover_clips" if bool(USE_TAKEOVER_CLIPS) else f"stride={SAMPLE_STRIDE}"

fig, ax = plt.subplots()


plot_cdf(
    ax,
    minade_planner,
    label=f"oracle@{planner_sets.shape[1]} over planner candidates ({snapshot_mode})",
)


plot_cdf(
    ax, minade_human_kmeans, label=f"human KMeans prototypes (global, K={human_centers.shape[0]})"
)


plot_cdf(ax, minade_grid, label=f"grid prototypes (global, K={grid_protos.shape[0]})")


if minade_external is not None:
    plot_cdf(
        ax, minade_external, label=f"external npy prototypes (global, K={external_protos.shape[0]})"
    )


ax.set_xlabel("min-ADE to executed trajectory (lower is better)")
ax.set_ylabel("CDF")
ax.set_title(f"Coverage of executed trajectories by different candidate sets ({snapshot_mode})")
ax.legend()
plt.show()


def summarize(name: str, arr: np.ndarray):
    q = np.quantile(arr, [0.1, 0.5, 0.9])
    print(f"{name:28s} mean={arr.mean():.3f}  p10={q[0]:.3f}  p50={q[1]:.3f}  p90={q[2]:.3f}")


summarize("planner", minade_planner)
summarize("human_kmeans", minade_human_kmeans)
summarize("grid", minade_grid)
if minade_external is not None:
    summarize("external", minade_external)

## How to read the CDF plots (and why the legends looked inconsistent)

### What is being plotted?
Each curve is a **CDF of min-ADE**. For each evaluated snapshot (one timestep / clip time), we compute:

\[
\text{minADE}(s) = \min_{i \in \{1..K\}} \; ADE(\hat{\tau}_i(s), \tau_{\text{exec}}(s))
\]

- \(\tau_{\text{exec}}\): executed future trajectory from `odom.jsonl` (robot frame)
- \(\hat{\tau}_i\): the i-th candidate trajectory
- **Lower is better**.

A point like **CDF = 0.92 at minADE = 1.0** means:
- **92% of snapshots have at least one candidate within 1 m ADE** (over the horizon).

### Why does “oracle@64” look similar to “planner candidates (K=64)”?
They are the **same quantity**.

- The curve formerly labeled "planner candidates (K=64)" is literally **oracle@64 over planner candidates**.
- Later panels call it explicitly **oracle@64**.

So there is no discrepancy — it was just naming.

### Why does the `planner_v2` curve look much worse even when the panel says K=64?
`planner_v2` applies **NMS + prob-thresholding**, which collapses the visible set to **~2 unique candidates on average**.
So even in a `K=64` panel, its **effective** candidate set size is tiny, making the oracle lower bound much worse.

### Snapshot subsets matter
Some parts evaluate:
- **stride-based per-tick samples**, or
- **takeover clip times** (recommended; matches Task 2).

The notebook now prints `snapshot_mode` in plot titles/legends so it’s clear which subset you’re looking at.

In [ ]:
# --- Planner candidate postprocess ablation: NMS / prob-threshold vs raw ---

# This mirrors `slow_brain_fast_planner.benchmarks.p1_tasks.planner_postprocess_v2`
# (endpoint-NMS + softmax + prob threshold).


def _softmax_stable_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    if x.size == 0:
        return x
    x = x - np.max(x)
    ex = np.exp(x)
    s = np.sum(ex)
    if s <= 0 or (not np.isfinite(s)):
        return np.full_like(ex, 1.0 / float(ex.size))
    return ex / s


def _trajectory_nms_endpoints_np(
    scores: np.ndarray,
    endpoints_xy: np.ndarray,
    *,
    max_trajectories: int,
    distance_threshold: float,
    pad_to_k: bool,
) -> list[int]:
    K = int(scores.shape[0])
    if K <= 0:
        return []

    # Deterministic tie-break: higher score first, then lower index.
    sorted_indices = sorted(range(K), key=lambda i: (-float(scores[i]), int(i)))

    keep: list[int] = []
    for i in sorted_indices:
        if len(keep) >= int(max_trajectories):
            break
        should_keep = True
        for kept_idx in keep:
            dx = float(endpoints_xy[i, 0] - endpoints_xy[kept_idx, 0])
            dy = float(endpoints_xy[i, 1] - endpoints_xy[kept_idx, 1])
            dist = math.sqrt(dx * dx + dy * dy)
            if dist < float(distance_threshold):
                should_keep = False
                break
        if should_keep:
            keep.append(int(i))

    if not keep:
        keep = [int(sorted_indices[0])]

    if pad_to_k:
        while len(keep) < int(max_trajectories):
            keep.append(int(keep[-1]))

    return keep[: int(max_trajectories)]


def planner_postprocess_v2_indices_np(
    scores: np.ndarray,
    endpoints_xy: np.ndarray,
    *,
    max_trajectories: int = 6,
    distance_threshold: float = 2.0,
    prob_threshold: float = 0.01,
    pad_to_k: bool = True,
):
    keep = _trajectory_nms_endpoints_np(
        scores,
        endpoints_xy,
        max_trajectories=int(max_trajectories),
        distance_threshold=float(distance_threshold),
        pad_to_k=bool(pad_to_k),
    )
    if not keep:
        return [], []

    probs = _softmax_stable_np(scores[np.asarray(keep, dtype=np.int64)])
    filtered = [
        (int(keep[i]), float(probs[i]))
        for i in range(len(keep))
        if float(probs[i]) >= float(prob_threshold)
    ]
    if not filtered:
        return [], []

    # Sort by probability (descending), stable tie-break by index.
    filtered.sort(key=lambda x: (-x[1], x[0]))
    return [i for i, _ in filtered], [p for _, p in filtered]


def build_per_sample_candidate_set(
    planner_sets: np.ndarray,
    planner_scores: np.ndarray,
    *,
    mode: str,
    out_k: int,
    nms_distance_threshold: float = 2.0,
    prob_threshold: float = 0.01,
    pad_to_k: bool = True,
):
    """Return: candidates (N,out_k,T,2), k_eff (N,), unique_eff (N,)"""
    assert planner_sets.ndim == 4
    N, K, T, _ = planner_sets.shape
    assert planner_scores.shape == (N, K)

    out = np.empty((N, int(out_k), T, 2), dtype=np.float32)
    k_eff = np.zeros((N,), dtype=np.int32)
    uniq_eff = np.zeros((N,), dtype=np.int32)

    for n in tqdm(range(N), desc=f"build_set:{mode}"):
        trajs = planner_sets[n]
        scores = planner_scores[n].astype(np.float64)

        if mode == "raw_all":
            idxs = list(range(K))
        elif mode == "topk_score":
            idxs = list(np.argsort(-scores)[: int(out_k)].astype(int))
        elif mode == "nms_only":
            # Endpoint-distance NMS (keep up to out_k), *no probability filtering*.
            endpoints = trajs[:, -1, :].astype(np.float64)
            idxs = _trajectory_nms_endpoints_np(
                scores,
                endpoints,
                max_trajectories=int(out_k),
                distance_threshold=float(nms_distance_threshold),
                pad_to_k=bool(pad_to_k),
            )
        elif mode == "planner_v2":
            # NMS + softmax + probability thresholding (what Task 2 overlay calls planner_v2).
            endpoints = trajs[:, -1, :].astype(np.float64)
            idxs, _probs = planner_postprocess_v2_indices_np(
                scores,
                endpoints,
                max_trajectories=int(out_k),
                distance_threshold=float(nms_distance_threshold),
                prob_threshold=float(prob_threshold),
                pad_to_k=bool(pad_to_k),
            )
        else:
            raise ValueError(f"unknown mode: {mode}")

        # If postprocess filters everything, fall back to global score argmax (keeps arrays finite).
        if not idxs:
            idx0 = int(np.argmax(scores)) if np.all(np.isfinite(scores)) else 0
            idxs = [idx0]

        k_eff[n] = int(len(idxs))
        uniq_eff[n] = int(len(set(int(i) for i in idxs)))

        # Pad to out_k for vectorized minADE.
        if len(idxs) < int(out_k):
            idxs = idxs + [idxs[-1]] * (int(out_k) - len(idxs))
        else:
            idxs = idxs[: int(out_k)]

        out[n] = trajs[np.asarray(idxs, dtype=np.int64)]

    return out, k_eff, uniq_eff


# Defaults used by the benchmark today (see `src/slow_brain_fast_planner/constants.py`)
DEFAULT_NMS_K = 6
DEFAULT_NMS_DIST = 2.0
DEFAULT_PROB_THR = 0.01

cand_raw6, k_raw6, u_raw6 = build_per_sample_candidate_set(
    planner_sets,
    planner_scores,
    mode="topk_score",
    out_k=DEFAULT_NMS_K,
)

cand_nms_only, k_nms, u_nms = build_per_sample_candidate_set(
    planner_sets,
    planner_scores,
    mode="nms_only",
    out_k=DEFAULT_NMS_K,
    nms_distance_threshold=DEFAULT_NMS_DIST,
    pad_to_k=True,
)

cand_planner_v2, k_v2, u_v2 = build_per_sample_candidate_set(
    planner_sets,
    planner_scores,
    mode="planner_v2",
    out_k=DEFAULT_NMS_K,
    nms_distance_threshold=DEFAULT_NMS_DIST,
    prob_threshold=DEFAULT_PROB_THR,
    pad_to_k=True,
)

minade_raw64 = minade_per_sample_candidates(executed_trajs, planner_sets)
minade_top6 = minade_per_sample_candidates(executed_trajs, cand_raw6)
minade_nms6 = minade_per_sample_candidates(executed_trajs, cand_nms_only)
minade_v2 = minade_per_sample_candidates(executed_trajs, cand_planner_v2)

fig, ax = plt.subplots()
plot_cdf(ax, minade_raw64, label="oracle@64 (raw candidates)")
plot_cdf(ax, minade_top6, label="top-6 by score (K=6)")
plot_cdf(ax, minade_nms6, label=f"NMS-only (K=6, dist={DEFAULT_NMS_DIST})")
plot_cdf(
    ax,
    minade_v2,
    label=f"NMS+prob-threshold (planner_v2; dist={DEFAULT_NMS_DIST}, p>={DEFAULT_PROB_THR})",
)
ax.set_xlabel("min-ADE to executed trajectory (lower is better)")
ax.set_ylabel("CDF")
ax.set_title("Effect of NMS vs NMS+prob-threshold on candidate-set quality")
ax.legend()
plt.show()

summarize("oracle@64", minade_raw64)
summarize("top6", minade_top6)
summarize("nms_only", minade_nms6)
summarize("planner_v2", minade_v2)

print("\nEffective candidate counts:")
print(f"- top6           mean={k_raw6.mean():.2f} unique_mean={u_raw6.mean():.2f}")
print(f"- nms_only        mean={k_nms.mean():.2f} unique_mean={u_nms.mean():.2f}")
print(f"- planner_v2      mean={k_v2.mean():.2f} unique_mean={u_v2.mean():.2f}")


# --- Sweep NMS distance threshold to see where the degradation comes from ---


def sweep_distance_threshold(distances=(0.25, 0.5, 0.75, 1.0, 1.5, 2.0)):
    rows = []
    for d in distances:
        cand_v2_d, k_d, u_d = build_per_sample_candidate_set(
            planner_sets,
            planner_scores,
            mode="planner_v2",
            out_k=DEFAULT_NMS_K,
            nms_distance_threshold=float(d),
            prob_threshold=DEFAULT_PROB_THR,
            pad_to_k=True,
        )
        m = minade_per_sample_candidates(executed_trajs, cand_v2_d)
        rows.append(
            {
                "dist": float(d),
                "minade_mean": float(m.mean()),
                "minade_p50": float(np.quantile(m, 0.5)),
                "k_eff_mean": float(k_d.mean()),
                "unique_eff_mean": float(u_d.mean()),
            }
        )
    return rows


rows = sweep_distance_threshold()
for r in rows:
    print(
        f"dist={r['dist']:<4.2f}  minADE_mean={r['minade_mean']:.3f}  p50={r['minade_p50']:.3f}  "
        f"k_eff_mean={r['k_eff_mean']:.2f}  unique_mean={r['unique_eff_mean']:.2f}"
    )

fig, ax = plt.subplots()
ax.plot([r["dist"] for r in rows], [r["minade_mean"] for r in rows], marker="o")
ax.set_xlabel("NMS distance threshold (m)")
ax.set_ylabel("mean min-ADE after planner_v2 postprocess")
ax.set_title("Sensitivity of postprocessed set quality to endpoint-NMS distance")
plt.show()

In [ ]:
# --- Alternative VLM-friendly candidate-set construction: endpoint k-center diversity ---

# Greedy k-center on endpoints (start from score argmax, then pick farthest endpoint each time).
# This selects a *subset of existing trajectories*; the output is still full trajectories,
# not straight lines.

# Toggle heavy plotting (k-center@32/@64 can take time).
RUN_CDF_GRIDS = True
RUN_MEAN_CURVES = True


def kcenter_endpoints_indices(
    scores: np.ndarray, endpoints_xy: np.ndarray, out_k: int
) -> list[int]:
    K = int(scores.shape[0])
    if K <= 0:
        return []

    if np.all(np.isfinite(scores)):
        start = int(np.argmax(scores))
    else:
        start = 0

    out_k = int(min(int(out_k), int(K)))

    selected = [start]
    selected_set = {start}

    for _ in range(1, int(out_k)):
        best_i = None
        best_d2 = -1.0
        for i in range(K):
            if i in selected_set:
                continue
            # distance to closest selected endpoint
            min_d2 = float("inf")
            for j in selected:
                dx = float(endpoints_xy[i, 0] - endpoints_xy[j, 0])
                dy = float(endpoints_xy[i, 1] - endpoints_xy[j, 1])
                d2 = dx * dx + dy * dy
                if d2 < min_d2:
                    min_d2 = d2
            if min_d2 > best_d2:
                best_d2 = min_d2
                best_i = int(i)

        if best_i is None:
            best_i = int(selected[-1])
        selected.append(int(best_i))
        selected_set.add(int(best_i))

    return selected[: int(out_k)]


def build_kcenter_candidate_set(
    planner_sets: np.ndarray, planner_scores: np.ndarray, *, out_k: int
):
    N, K, T, _ = planner_sets.shape
    out_k = int(out_k)

    # Shortcut: if out_k >= K, k-center is just the full set.
    if out_k >= int(K):
        return np.asarray(planner_sets, dtype=np.float32)

    out = np.empty((N, out_k, T, 2), dtype=np.float32)
    for n in tqdm(range(N), desc=f"build_set:kcenter@{out_k}"):
        trajs = planner_sets[n]
        scores = planner_scores[n]
        endpoints = trajs[:, -1, :]
        idxs = kcenter_endpoints_indices(
            scores.astype(np.float64), endpoints.astype(np.float64), out_k
        )
        if not idxs:
            idxs = [0]
        if len(idxs) < out_k:
            idxs = idxs + [idxs[-1]] * (out_k - len(idxs))
        out[n] = trajs[np.asarray(idxs[:out_k], dtype=np.int64)]
    return out


def cdf_grid_for_ks(
    *,
    planner_sets: np.ndarray,
    planner_scores: np.ndarray,
    title_prefix: str,
    ks=(6, 12, 24, 32, 64),
):
    ks = [int(k) for k in ks]

    # Baseline: oracle over the full set
    minade_raw64 = minade_per_sample_candidates(executed_trajs, planner_sets)

    ncols = 3
    nrows = int(math.ceil(len(ks) / ncols))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols * 5.2, nrows * 4.6))
    axes = np.asarray(axes).reshape(-1)

    for ax, k in zip(axes, ks, strict=False):
        cand_top = build_per_sample_candidate_set(
            planner_sets, planner_scores, mode="topk_score", out_k=k
        )[0]
        cand_nms, _k_eff_nms, u_eff_nms = build_per_sample_candidate_set(
            planner_sets,
            planner_scores,
            mode="nms_only",
            out_k=k,
            nms_distance_threshold=DEFAULT_NMS_DIST,
            pad_to_k=True,
        )
        cand_v2, _k_eff_v2, u_eff_v2 = build_per_sample_candidate_set(
            planner_sets,
            planner_scores,
            mode="planner_v2",
            out_k=k,
            nms_distance_threshold=DEFAULT_NMS_DIST,
            prob_threshold=DEFAULT_PROB_THR,
            pad_to_k=True,
        )
        cand_kc = build_kcenter_candidate_set(planner_sets, planner_scores, out_k=k)

        m_top = minade_per_sample_candidates(executed_trajs, cand_top)
        m_nms = minade_per_sample_candidates(executed_trajs, cand_nms)
        m_v2 = minade_per_sample_candidates(executed_trajs, cand_v2)
        m_kc = minade_per_sample_candidates(executed_trajs, cand_kc)

        eff_uniq_nms = float(np.mean(u_eff_nms)) if u_eff_nms is not None else float("nan")
        eff_uniq_v2 = float(np.mean(u_eff_v2)) if u_eff_v2 is not None else float("nan")

        plot_cdf(ax, minade_raw64, label="oracle@64")
        plot_cdf(ax, m_top, label=f"topK_score (K={k})")
        plot_cdf(ax, m_kc, label=f"kcenter_endpt (K={k})")
        plot_cdf(ax, m_nms, label=f"nms_only (uniq≈{eff_uniq_nms:.1f})")
        plot_cdf(ax, m_v2, label=f"nms+prob (uniq≈{eff_uniq_v2:.1f})")

        ax.set_title(f"{title_prefix}: quality @ K={k}")
        ax.set_xlabel("min-ADE (lower is better)")
        ax.set_ylabel("CDF")
        ax.legend(fontsize=9)

    # Turn off unused axes.
    for j in range(len(ks), len(axes)):
        axes[j].axis("off")

    fig.suptitle(f"{title_prefix}: k-center vs score-topK vs planner_v2", y=1.02)
    plt.tight_layout()
    plt.show()


def mean_minade_vs_k(
    *,
    planner_sets: np.ndarray,
    planner_scores: np.ndarray,
    title_prefix: str,
    ks=(6, 12, 24, 32, 64),
):
    ks = [int(k) for k in ks]
    rows = []

    for k in ks:
        cand_top = build_per_sample_candidate_set(
            planner_sets, planner_scores, mode="topk_score", out_k=k
        )[0]
        cand_nms, _k_eff_nms, u_nms = build_per_sample_candidate_set(
            planner_sets,
            planner_scores,
            mode="nms_only",
            out_k=k,
            nms_distance_threshold=DEFAULT_NMS_DIST,
            pad_to_k=True,
        )
        cand_v2, _k_eff_v2, u_v2 = build_per_sample_candidate_set(
            planner_sets,
            planner_scores,
            mode="planner_v2",
            out_k=k,
            nms_distance_threshold=DEFAULT_NMS_DIST,
            prob_threshold=DEFAULT_PROB_THR,
            pad_to_k=True,
        )
        cand_kc = build_kcenter_candidate_set(planner_sets, planner_scores, out_k=k)

        m_raw = minade_per_sample_candidates(executed_trajs, planner_sets)
        m_top = minade_per_sample_candidates(executed_trajs, cand_top)
        m_nms = minade_per_sample_candidates(executed_trajs, cand_nms)
        m_v2 = minade_per_sample_candidates(executed_trajs, cand_v2)
        m_kc = minade_per_sample_candidates(executed_trajs, cand_kc)

        rows.append(
            {
                "K": k,
                "oracle64_mean": float(m_raw.mean()),
                "topK_mean": float(m_top.mean()),
                "kcenter_mean": float(m_kc.mean()),
                "nms_only_mean": float(m_nms.mean()),
                "nms_only_unique_mean": float(np.mean(u_nms)),
                "nms_prob_mean": float(m_v2.mean()),
                "nms_prob_unique_mean": float(np.mean(u_v2)),
            }
        )

    print(f"\n[{title_prefix}] Mean min-ADE summary")
    print("K\toracle@64\ttopK\tkcenter\tnms_only\tnms_only_uniq\tnms+prob\tnms+prob_uniq")
    for r in rows:
        print(
            f"{r['K']}\t{r['oracle64_mean']:.4f}\t\t{r['topK_mean']:.4f}\t\t{r['kcenter_mean']:.4f}\t\t"
            f"{r['nms_only_mean']:.4f}\t\t{r['nms_only_unique_mean']:.2f}\t\t{r['nms_prob_mean']:.4f}\t\t{r['nms_prob_unique_mean']:.2f}"
        )

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot([r["K"] for r in rows], [r["topK_mean"] for r in rows], marker="o", label="topK_score")
    ax.plot(
        [r["K"] for r in rows], [r["kcenter_mean"] for r in rows], marker="o", label="kcenter_endpt"
    )
    ax.plot(
        [r["K"] for r in rows], [r["nms_only_mean"] for r in rows], marker="o", label="nms_only"
    )
    ax.plot(
        [r["K"] for r in rows], [r["nms_prob_mean"] for r in rows], marker="o", label="nms+prob"
    )
    ax.axhline(rows[0]["oracle64_mean"], linestyle="--", color="k", alpha=0.5, label="oracle@64")
    ax.set_xscale("log", base=2)
    ax.set_xticks(list(ks))
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax.set_xlabel("K")
    ax.set_ylabel("mean min-ADE")
    ax.set_title(f"{title_prefix}: mean min-ADE vs K")
    ax.legend()
    plt.show()

    return rows


# --- Final, decision-ready comparison: prelogged vs goal-less ONNX on the same snapshot times ---

KS_MAIN = (6, 9, 12, 24, 32, 64)

if RUN_CDF_GRIDS:
    cdf_grid_for_ks(
        planner_sets=planner_sets,
        planner_scores=planner_scores,
        title_prefix="prelogged",
        ks=KS_MAIN,
    )
    if planner_sets_goal_less is not None and planner_scores_goal_less is not None:
        cdf_grid_for_ks(
            planner_sets=planner_sets_goal_less,
            planner_scores=planner_scores_goal_less,
            title_prefix="goal_less_onnx",
            ks=KS_MAIN,
        )

if RUN_MEAN_CURVES:
    rows_pre = mean_minade_vs_k(
        planner_sets=planner_sets,
        planner_scores=planner_scores,
        title_prefix="prelogged",
        ks=KS_MAIN,
    )
    if planner_sets_goal_less is not None and planner_scores_goal_less is not None:
        rows_gl = mean_minade_vs_k(
            planner_sets=planner_sets_goal_less,
            planner_scores=planner_scores_goal_less,
            title_prefix="goal_less_onnx",
            ks=KS_MAIN,
        )

print("\n=== Quick recommendation (oracle proxy; validate with real VLM) ===")
if planner_sets_goal_less is not None:
    print(
        "- If VLM can only handle ~6 candidates: compare goal-less ONNX and prelogged "
        "with kcenter_endpt (K=6)."
    )
if planner_sets_goal_less is not None:
    print(
        "- If VLM can handle 12: prelogged + kcenter_endpt (K=12) is very strong; "
        "goal-less + kcenter_endpt (K=12) is also competitive."
    )
print(
    "- Avoid planner_v2 prob-thresholding as the candidate set: "
    "it collapses to ~2 unique candidates on average."
)

## Notes on using these candidates for Task 2 (decision guide)

- **Prelogged (goal-conditioned) 64-candidate sets** usually have the best *raw64* coverage (oracle@64).
- **Goal-less ONNX 64-candidate sets** can be *more diverse*; on this dataset they are worse at raw64 coverage, but can be **better after k-center downselection at small K**.
- **Planner_v2 (NMS + prob-threshold)** is a poor *candidate-set preparation* step for VLM selection: it often collapses to ~2 unique candidates, which raises the oracle lower bound drastically.

Practical guidance:
- If the VLM can only handle **K≈6**, try **k-center endpoints** (both prelogged and goal-less).
- If the VLM can handle **K≈12**, **prelogged + k-center** is a very strong default (and goal-less + k-center is still competitive).
- Use this notebook’s **CDF / mean-minADE vs K** as an *oracle proxy*; the final answer still requires running the real VLM.

Global prototype sets (human KMeans / grid / external):
- These are *context-agnostic* but can be useful if you want a fixed discrete action space.
- If you want to try "VLM selects from global prototypes" in Task 2, you likely want:
  - **A fixed candidate set** `K x T x 2` (e.g. `human_centers`)
  - oracle evaluation (min-ADE/min-FDE) against those candidates, the same way we evaluate planner candidates.
